# QLoRA Fine-Tuning for SMS Spam Detection: Base vs Fine-Tuned

**Task:** Fine-tune a small language model with **QLoRA** on the **SMS Spam Collection** dataset to classify messages as `spam` or `ham` (legitimate). Then compare the **base** and **fine-tuned** models on the **same held-out test set** using **spam precision, recall and F1**.

| Item | Choice |
|---|---|
| Model | `Qwen/Qwen2.5-0.5B-Instruct` (small, open, ungated) |
| Method | QLoRA: 4-bit NF4 quantised base + LoRA adapters (r=16) |
| Data | UCI SMS Spam Collection (5,574 messages), exact duplicates removed |
| Split | 80% train / 10% validation / 10% test, stratified, seed 42 |
| Positive class | `spam` (all precision/recall/F1 numbers are for spam) |

**How to run on Google Colab:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. Total time is roughly 10 to 15 minutes on a free T4.

### Fair comparison
- Both models see the **identical prompt** and the **identical test set**.
- "Base" = the same 4-bit Qwen model **before** any adapter is attached (zero-shot). "Fine-tuned" = that model **plus** the trained LoRA adapter. So the only difference is the adapter.
- Both are scored the same way: we compare the model's next-token logit for `spam` against `ham`. This avoids unreliable text parsing of free-form generations, so the base model is not penalised for formatting mistakes.

## 1. Setup

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets scikit-learn

In [ ]:
import io, zipfile, urllib.request, random, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             confusion_matrix, roc_auc_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

assert torch.cuda.is_available(), "No GPU found. Use Runtime > Change runtime type > T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

# bfloat16 needs an Ampere+ GPU (compute capability >= 8). The free Colab T4 is 7.5, so use float16 there.
# (torch.cuda.is_bf16_supported() can say True on a T4 because of slow emulation, so we check capability.)
USE_BF16 = torch.cuda.get_device_capability(0)[0] >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print("Compute dtype:", COMPUTE_DTYPE)

## 2. Data
We load the SMS Spam Collection from the Hugging Face Hub. If that fails, we fall back to the original UCI zip file.

Duplicate message texts are removed **before** splitting. The raw dataset contains hundreds of exact duplicates, and if a copy lands in both train and test, the test score would be inflated by memorisation.

In [ ]:
def load_sms():
    try:
        from datasets import load_dataset
        ds = load_dataset("ucirvine/sms_spam", split="train")
        df = ds.to_pandas().rename(columns={"sms": "text"})
        df["label"] = df["label"].map({0: "ham", 1: "spam"})
        print("Loaded from Hugging Face Hub")
    except Exception as e:
        print("HF load failed (", repr(e)[:100], ") -> using UCI zip")
        url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
        raw = urllib.request.urlopen(url).read()
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            name = [n for n in z.namelist() if n.endswith("SMSSpamCollection")][0]
            df = pd.read_csv(z.open(name), sep="\t", header=None, names=["label", "text"])
    df["text"] = df["text"].astype(str).str.strip()
    return df[["text", "label"]]

raw_df = load_sms()
print("Raw rows:", len(raw_df))
df = raw_df.drop_duplicates(subset="text").reset_index(drop=True)
print("After removing exact duplicate texts:", len(df))
print(df["label"].value_counts().to_string())
df.sample(5, random_state=SEED)

In [ ]:
trainval_df, test_df = train_test_split(df, test_size=0.10, stratify=df["label"], random_state=SEED)
train_df, val_df = train_test_split(trainval_df, test_size=1/9, stratify=trainval_df["label"], random_state=SEED)
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]

summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "messages": [len(train_df), len(val_df), len(test_df)],
    "spam": [(d["label"] == "spam").sum() for d in (train_df, val_df, test_df)],
    "ham": [(d["label"] == "ham").sum() for d in (train_df, val_df, test_df)],
})
summary["spam %"] = (100 * summary["spam"] / summary["messages"]).round(1)
assert not (set(train_df.text) & set(test_df.text)), "train/test leakage!"
summary

## 3. Prompt format and scoring
The model is given a chat-formatted instruction and should answer with one word: `spam` or `ham`.

For evaluation we do **not** generate text. We take the logits of the next token and predict `spam` when `logit("spam") > logit("ham")`. The same function scores both the base and the fine-tuned model.

In [ ]:
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

SYSTEM = ("You are an SMS spam filter. Classify the SMS as 'spam' (unwanted promotions, "
          "prizes, scams, phishing, premium-rate services) or 'ham' (a legitimate, normal "
          "message). Answer with exactly one word: spam or ham.")

def build_prompt(text):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": f"SMS: {text[:600]}"}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

SPAM_ID = tok("spam", add_special_tokens=False).input_ids[0]
HAM_ID = tok("ham", add_special_tokens=False).input_ids[0]
assert SPAM_ID != HAM_ID, "first tokens of the two labels must differ"
print("spam token:", repr(tok.decode([SPAM_ID])), "| ham token:", repr(tok.decode([HAM_ID])))
print("\n--- example prompt ---\n" + build_prompt(train_df.text[0]))

In [ ]:
@torch.no_grad()
def score_messages(model, texts, batch_size=16):
    """Return margin = logit(spam) - logit(ham) at the first answer position."""
    model.eval()
    tok.padding_side = "left"
    texts = list(texts)
    order = np.argsort([len(t) for t in texts])      # sort by length -> less padding
    margins = np.zeros(len(texts), dtype=np.float32)
    for i in range(0, len(texts), batch_size):
        idx = order[i:i + batch_size]
        enc = tok([build_prompt(texts[j]) for j in idx], return_tensors="pt",
                  padding=True, add_special_tokens=False).to(model.device)
        last = model(**enc).logits[:, -1, :].float()
        margins[idx] = (last[:, SPAM_ID] - last[:, HAM_ID]).cpu().numpy()
    return margins

def metrics_from_margins(margins, y_true):
    y_pred = (margins > 0).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary",
                                                 pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    # ROC-AUC is threshold-free: it shows whether the model *ranks* spam above ham,
    # independent of where the spam/ham cut-off (margin > 0) happens to sit.
    return {"precision": p, "recall": r, "f1": f, "accuracy": accuracy_score(y_true, y_pred),
            "roc_auc": roc_auc_score(y_true, margins),
            "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn)}

y_val = (val_df["label"] == "spam").astype(int).values
y_test = (test_df["label"] == "spam").astype(int).values

## 4. Load the 4-bit quantised base model and evaluate it (zero-shot)
This is the **base model** result. It is computed on the held-out test set **before** any training happens.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config,
                                             device_map={"": 0})
model.config.use_cache = False
print(f"Base model memory: {model.get_memory_footprint() / 1e6:.0f} MB")

In [ ]:
t0 = time.time()
base_test_margins = score_messages(model, test_df["text"])
base_metrics = metrics_from_margins(base_test_margins, y_test)
print(f"Scored {len(test_df)} test messages in {time.time() - t0:.0f}s")
print({k: round(v, 4) if isinstance(v, float) else v for k, v in base_metrics.items()})

## 5. QLoRA fine-tuning
LoRA adapters (rank 16) are attached to all attention and MLP projection layers of the frozen 4-bit model. Only the adapters train.

The training target is the answer (`spam` / `ham` + end-of-turn token). The loss is computed **only on the answer tokens**, not on the prompt.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
EOT_ID = tok.convert_tokens_to_ids("<|im_end|>")

def encode_example(text, label):
    prompt_ids = tok(build_prompt(text), add_special_tokens=False).input_ids
    answer_ids = tok(label, add_special_tokens=False).input_ids + [EOT_ID]
    return {"input_ids": prompt_ids + answer_ids,
            "labels": [-100] * len(prompt_ids) + answer_ids}     # loss on answer only

class SMSDataset(torch.utils.data.Dataset):
    def __init__(self, frame):
        self.items = [encode_example(t, l) for t, l in zip(frame["text"], frame["label"])]
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def collate(batch):                       # right-pad for training
    L = max(len(b["input_ids"]) for b in batch)
    ids, lab, att = [], [], []
    for b in batch:
        pad = L - len(b["input_ids"])
        ids.append(b["input_ids"] + [tok.pad_token_id] * pad)
        lab.append(b["labels"] + [-100] * pad)
        att.append([1] * len(b["input_ids"]) + [0] * pad)
    return {"input_ids": torch.tensor(ids), "labels": torch.tensor(lab),
            "attention_mask": torch.tensor(att)}

train_ds, val_ds = SMSDataset(train_df), SMSDataset(val_df)
print("train/val examples:", len(train_ds), len(val_ds),
      "| longest sequence:", max(len(x["input_ids"]) for x in train_ds.items))

In [ ]:
args = TrainingArguments(
    output_dir="qlora-sms-spam",
    num_train_epochs=2,
    per_device_train_batch_size=4,          # small: Qwen's 152k vocab makes logits memory-heavy
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,          # effective batch size 4 x 8 = 32
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,                        # ~5% of training (warmup_ratio was removed in new transformers)
    weight_decay=0.0,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
    seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=collate)
import gc; gc.collect(); torch.cuda.empty_cache()    # free memory cached by the base-model evaluation
t0 = time.time()
train_result = trainer.train()
print(f"\nTraining finished in {(time.time() - t0) / 60:.1f} min | final train loss: {train_result.training_loss:.4f}")

In [ ]:
log = pd.DataFrame(trainer.state.log_history)
fig, ax = plt.subplots(figsize=(6, 3.2))
tr = log.dropna(subset=["loss"])
ax.plot(tr["step"], tr["loss"], label="train loss")
ev = log.dropna(subset=["eval_loss"])
ax.scatter(ev["step"], ev["eval_loss"], color="C3", zorder=3, label="validation loss")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.set_yscale("log"); ax.legend(); ax.set_title("QLoRA training")
plt.tight_layout(); plt.show()

## 6. Evaluate the fine-tuned model on the same held-out test set
The adapter-equipped model is now scored with the **same function, prompt and test messages** as the base model.

In [ ]:
torch.cuda.empty_cache()
ft_test_margins = score_messages(model, test_df["text"], batch_size=8)
ft_metrics = metrics_from_margins(ft_test_margins, y_test)
ft_val_metrics = metrics_from_margins(score_messages(model, val_df["text"], batch_size=8), y_val)
print("Fine-tuned TEST:", {k: round(v, 4) if isinstance(v, float) else v for k, v in ft_metrics.items()})
print("Fine-tuned VAL :", {k: round(v, 4) if isinstance(v, float) else v for k, v in ft_val_metrics.items()})

## 7. Results: base vs fine-tuned (spam class, same test set)

In [ ]:
results = pd.DataFrame({"Base (zero-shot)": base_metrics, "Fine-tuned (QLoRA)": ft_metrics}).T
results["Δ F1 vs base"] = ["", f"{ft_metrics['f1'] - base_metrics['f1']:+.4f}"]
show = results.copy()
for c in ["precision", "recall", "f1", "accuracy", "roc_auc"]:
    show[c] = show[c].astype(float).round(4)
for c in ["TP", "FP", "FN", "TN"]:
    show[c] = show[c].astype(int)
print(f"Test set: {len(test_df)} messages, {y_test.sum()} spam\n")
display(show[["precision", "recall", "f1", "accuracy", "roc_auc", "TP", "FP", "FN", "TN", "Δ F1 vs base"]])
results.drop(columns="Δ F1 vs base").to_csv("results_base_vs_finetuned.csv")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

# (a) grouped bars
names = ["precision", "recall", "f1"]
x = np.arange(len(names)); w = 0.36
b1 = axes[0].bar(x - w/2, [base_metrics[n] for n in names], w, label="Base")
b2 = axes[0].bar(x + w/2, [ft_metrics[n] for n in names], w, label="Fine-tuned")
for bars in (b1, b2):
    for b in bars:
        axes[0].text(b.get_x() + b.get_width()/2, b.get_height() + 0.01, f"{b.get_height():.2f}", ha="center", fontsize=9)
axes[0].set_xticks(x); axes[0].set_xticklabels(["Precision", "Recall", "F1"])
axes[0].set_ylim(0, 1.12); axes[0].set_title("Spam-class metrics (test set)"); axes[0].legend()

# (b, c) confusion matrices
for ax, (title, m) in zip(axes[1:], [("Base", base_metrics), ("Fine-tuned", ft_metrics)]):
    cm = np.array([[m["TN"], m["FP"]], [m["FN"], m["TP"]]])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["pred ham", "pred spam"]); ax.set_yticklabels(["true ham", "true spam"])
    ax.set_title(f"{title}: confusion matrix")
plt.tight_layout(); plt.show()

In [ ]:
# Auto-generated summary of the numbers above
def pct(v): return f"{100 * v:.1f}%"
print(f"Base       : precision {pct(base_metrics['precision'])}, recall {pct(base_metrics['recall'])}, F1 {pct(base_metrics['f1'])}")
print(f"Fine-tuned : precision {pct(ft_metrics['precision'])}, recall {pct(ft_metrics['recall'])}, F1 {pct(ft_metrics['f1'])}")
print(f"\nF1 change: {100 * (ft_metrics['f1'] - base_metrics['f1']):+.1f} points")
print(f"Spam caught : {base_metrics['TP']} -> {ft_metrics['TP']} of {y_test.sum()}")
print(f"False alarms (ham flagged as spam): {base_metrics['FP']} -> {ft_metrics['FP']}")
print(f"Missed spam: {base_metrics['FN']} -> {ft_metrics['FN']}")

## 8. Error analysis
Which test messages does the fine-tuned model still get wrong?

In [ ]:
err = test_df.copy()
err["margin"] = ft_test_margins
err["pred"] = np.where(err["margin"] > 0, "spam", "ham")
wrong = err[err["pred"] != err["label"]].copy()
wrong["error type"] = np.where(wrong["label"] == "spam", "missed spam (FN)", "false alarm (FP)")
pd.set_option("display.max_colwidth", 160)
print(f"{len(wrong)} errors out of {len(err)}")
wrong[["error type", "label", "pred", "text"]].reset_index(drop=True)

In [ ]:
# Save the trained adapter (small: a few MB)
model.save_pretrained("qlora-sms-spam-adapter")
tok.save_pretrained("qlora-sms-spam-adapter")
print("Adapter saved to ./qlora-sms-spam-adapter")

In [ ]:
# Zip the adapter and download it (needed for the FastAPI backend / UI). Colab only.
!zip -qr adapter.zip qlora-sms-spam-adapter
try:
    from google.colab import files
    files.download("adapter.zip")
except ImportError:
    print("Not on Colab: adapter.zip was created in the current folder.")

## 9. Discussion
*(Write your own interpretation here after running. Suggested points, using the numbers printed above.)*

- **What changed?** Compare precision and recall of the base and fine-tuned models. A zero-shot model usually has no idea of this dataset's notion of "spam" (e.g. promotional short codes, "txt WIN to 80086"), so it tends to either over-flag or under-flag. Fine-tuning calibrates it.
- **Precision vs recall trade-off.** Precision = of the messages flagged as spam, how many really were. Recall = of all real spam, how many were caught. For a spam filter, a false positive (a real message hidden) is usually costlier than a missed spam, so precision matters a lot.
- **Why F1?** The data is imbalanced (about 13% spam), so accuracy alone is misleading: predicting "ham" for everything already gives about 87% accuracy. F1 balances precision and recall on the spam class.
- **Why QLoRA?** 4-bit quantisation plus LoRA lets us fine-tune on a free Colab GPU by training only a tiny fraction of the parameters (see `print_trainable_parameters` above).

### Limitations
- Single run with one seed and a test set of only about 500 messages (only a few dozen spam; the exact count is printed in Section 7), so differences of a point or two are within noise.
- The base model's spam/ham cut-off is simply `logit(spam) > logit(ham)`. A zero-shot model can be biased toward one word, which hurts its precision/recall even if it ranks messages well. The `roc_auc` column is threshold-free and shows that ranking ability separately.
- The dataset is old (2000s, UK/Singapore English SMS). Modern spam may differ.
- The base model is evaluated zero-shot with one fixed prompt; a different prompt or few-shot examples could change its score.